In [2]:
%pip install --quiet --upgrade langchain langchain-community pypdf docx2txt "unstructured[md]"

Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_community.document_loaders import PyPDFLoader

# 工作目录下有 'docs/sample.pdf'
loader = PyPDFLoader("docs/ai_model_agent_history_script.pdf")
pages = loader.load_and_split()

# 打印第一页的内容
print(pages[1].page_content)

1956 年达特茅斯 AI 夏季研究项⽬参与者⼾外合影
早期的⼈⼯智能主要以符号主义和规则系统为主，研究者们尝试⽤⼈⼯编写的规则让计算机解决
问题。例如，1966 年，⿇省理⼯学院的约瑟夫 · 魏泽鲍姆开发了聊天程序 ELIZA ，它通过模式匹配
和简单规则模拟⼼理医⽣的对话，成为世界上第⼀个聊天机器⼈[toloka.ai]。 ELIZA 虽然简单，但
展⽰了计算机理解和⽣成⾃然语⾔的潜⼒，被视为⾃然语⾔处理（ NLP ）的起点[toloka.ai]。
Eliza 聊天界⾯，模拟⽤⼾与 Eliza 对话
进⼊1980 年代，统计学习⽅法开始在 NLP 领域崭露头⻆。研究者们尝试让计算机从⼤量语料中⾃
动学习语⾔规律，例如1988 年IBM 的研究团队提出了基于统计的机器翻译⽅法，为后来的统计语
⾔模型奠定了基础[techtarget.com]。与此同时，神经⽹络的研究也在曲折中前进。1989 年，杨⽴
昆等⼈⽤卷积神经⽹络成功识别⼿写数字，展⽰了神经⽹络在实际问题中的潜⼒
[techtarget.com]。1997 年，霍赫赖特和施密德胡伯提出了⻓短期记忆⽹络（ LSTM ），解决了传统
循环神经⽹络的⻓期依赖问题，使得神经⽹络能够更好地处理⻓序列数据[techtarget.com]。
LSTM 的出现被视为深度学习在语⾔处理领域的重要突破[toloka.ai]。


In [ ]:
from langchain_community.document_loaders import Docx2txtLoader

# 假设工作目录下有 'docs/ai_model_agent_history_script.docx'
loader = Docx2txtLoader("docs/ai_model_agent_history_script.docx")
data = loader.load()

print(data[0].page_content)

In [9]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader

# 工作目录下有 'docs/test_markdown.md'
# 其内容可能为：
# # Markdown 图片处理
# 这是一个本地图片引用：
# ![本地图片示例](./images/simple.png)

loader = UnstructuredMarkdownLoader("docs/test_markdown.md")
data = loader.load()

print(data[0].page_content)

测试Markdown文档

这是一个用于测试多模态文档处理功能的Markdown文件。

图片示例

下面是一些图片引用的示例：

LangChain Logo

OpenAI Logo

本地图片引用

如果有本地图片文件，可以这样引用：

本地图片

另一个本地图片

文档内容

这个文档用于演示： 1. 图片引用的识别和提取 2. URL替换功能 3. 多模态文档处理流程

处理后，所有的图片引用都会被替换为COS存储的URL。


In [10]:
COS_SECRET_ID = "你的SecretId"
COS_SECRET_KEY = "你的SecretKey"
COS_BUCKET = "你的存储桶名称，例如 langchain-demo-1250000000"
COS_REGION = "你的存储桶所在地域，例如 ap-shanghai"

In [40]:
class MultimodalDocumentLoader:
    def __init__(self, cos_bucket: str = None, cos_region: str = None):
        try:
            # 尝试从 .env 加载配置
            load_dotenv()
            self.cos_secret_id = os.getenv("COS_SECRET_ID")
            self.cos_secret_key = os.getenv("COS_SECRET_KEY")
            self.cos_region = cos_region or os.getenv("COS_REGION")
            self.cos_bucket = cos_bucket or os.getenv("COS_BUCKET")
            self.cos_domain = os.getenv("COS_DOMAIN")

            # 检查关键配置是否存在
            if (
                self.cos_secret_id
                and self.cos_secret_key
                and self.cos_bucket
                and self.cos_region
            ):
                config = CosConfig(
                    Region=self.cos_region,
                    SecretId=self.cos_secret_id,
                    SecretKey=self.cos_secret_key,
                )
                self.cos_client = CosS3Client(config)
                self.cos_enabled = True
            else:
                raise ValueError("COS配置不完整")

        except Exception as e:
            self.cos_client = None
            self.cos_enabled = False

        # 构建基础URL
        if self.cos_enabled and self.cos_domain:
            self.base_url = f"https://{self.cos_domain}"
        elif self.cos_enabled:
            self.base_url = (
                f"https://{self.cos_bucket}.cos.{self.cos_region}.myqcloud.com"
            )
        else:
            # 模拟模式下的URL
            self.base_url = "file:///temp_images_simulated"

        # 创建本地临时目录
        self.temp_dir = Path("temp_images")
        self.temp_dir.mkdir(exist_ok=True)

In [41]:
from typing import Dict, List


def extract_images_from_pdf(self, pdf_path: str) -> List[Dict]:
    images = []
    doc = fitz.open(pdf_path)
    for page_num in range(len(doc)):
        for img in doc.get_page_images(page_num, full=True):
            xref = img[0]
            pix = fitz.Pixmap(doc, xref)
            if pix.n - pix.alpha < 4:  # 排除CMYK等复杂颜色空间的图片
                img_data = pix.tobytes("png")  # 统一转换为PNG格式
                img_hash = hashlib.md5(img_data).hexdigest()
                images.append(
                    {
                        "data": img_data,
                        "format": "png",
                        "page": page_num + 1,
                        "hash": img_hash,
                        "filename": f"pdf_page{page_num+1}_{img_hash[:8]}.png",
                    }
                )
    doc.close()
    return images

In [42]:
import zipfile
from pathlib import Path


def extract_images_from_word(self, word_path: str) -> List[Dict]:
    images = []
    with zipfile.ZipFile(word_path, "r") as docx:
        # 图片通常存储在 'word/media/' 目录下
        media_files = [f for f in docx.namelist() if f.startswith("word/media/")]
        for media_file in media_files:
            img_data = docx.read(media_file)
            file_ext = Path(media_file).suffix.lower()[1:]
            img_hash = hashlib.md5(img_data).hexdigest()
            images.append(
                {
                    "data": img_data,
                    "format": file_ext,
                    "original_path": media_file,
                    "hash": img_hash,
                    "filename": f"word_{Path(media_file).stem}_{img_hash[:8]}.{file_ext}",
                }
            )
    return images

In [43]:
import re


def extract_images_from_markdown(self, md_path: str) -> List[Dict]:
    images = []
    md_dir = Path(md_path).parent
    with open(md_path, "r", encoding="utf-8") as f:
        content = f.read()

    # 正则表达式匹配 ![alt_text](img_path)
    img_pattern = r"!\\[([^\\]]*)\\]\\(([^\\)]+)\\)"
    for match in re.finditer(img_pattern, content):
        alt_text = match.group(1)
        img_path_str = match.group(2)

        # 只处理本地图片，忽略网络图片
        if not img_path_str.startswith(("http://", "https://")):
            full_img_path = (md_dir / img_path_str).resolve()
            if full_img_path.exists():
                with open(full_img_path, "rb") as img_file:
                    img_data = img_file.read()

                file_ext = full_img_path.suffix.lower()[1:]
                img_hash = hashlib.md5(img_data).hexdigest()
                images.append(
                    {
                        "data": img_data,
                        "format": file_ext,
                        "alt_text": alt_text,
                        "original_path": img_path_str,
                        "hash": img_hash,
                        "filename": f"md_{full_img_path.stem}_{img_hash[:8]}.{file_ext}",
                        "markdown_syntax": match.group(0),  # 保存原始语法用于替换
                    }
                )
    return images

In [44]:
def upload_to_cos(self, image_data: bytes, filename: str) -> str:
    if self.cos_enabled:
        try:
            object_key = f"images/{filename}"
            self.cos_client.put_object(
                Bucket=self.cos_bucket,
                Body=image_data,
                Key=object_key,
                ContentType=self._get_content_type(filename),
            )
            cos_url = f"{self.base_url}/{object_key}"
            logger.info(f"✅ COS上传成功: {filename} -> {cos_url}")
            return cos_url
        except Exception as e:
            logger.error(f"❌ COS上传失败: {e}")
            # 上传失败时，也降级到模拟模式
            return self._simulate_cos_upload(image_data, filename)
    else:
        return self._simulate_cos_upload(image_data, filename)


def _simulate_cos_upload(self, image_data: bytes, filename: str) -> str:
    temp_file = self.temp_dir / filename
    temp_file.write_bytes(image_data)
    logger.info(f"🔄 模拟上传: {filename} 保存至本地")
    # 返回一个模拟的URL
    return f"{self.base_url}/{filename}"


def _get_content_type(self, filename: str) -> str:
    # 根据文件后缀返回MIME类型
    ext = Path(filename).suffix.lower()
    content_types = {
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".bmp": "image/bmp",
    }
    return content_types.get(ext, "application/octet-stream")

In [48]:
def process_document(self, file_path: str) -> Dict:
    file_path = Path(file_path)
    file_ext = file_path.suffix.lower()
    result = {
        "file_path": str(file_path),
        "images": [],
        "cos_urls": [],
        "processed_content": "",
    }

    # 1. 根据文件类型调用不同的提取函数
    if file_ext == ".pdf":
        images = self.extract_images_from_pdf(str(file_path))
    elif file_ext == ".docx":
        images = self.extract_images_from_word(str(file_path))
    elif file_ext == ".md":
        images = self.extract_images_from_markdown(str(file_path))
        result["original_content"] = file_path.read_text(encoding="utf-8")
    else:
        logger.warning(f"不支持的文件类型: {file_ext}")
        return result

    result["images"] = images

    # 2. 遍历提取的图片并上传
    for img in images:
        cos_url = self.upload_to_cos(img["data"], img["filename"])
        if cos_url:
            result["cos_urls"].append(
                {"filename": img["filename"], "cos_url": cos_url, "original_info": img}
            )

    # 3. 如果是Markdown，执行内容替换
    if file_ext == ".md":
        processed_content = result["original_content"]
        for url_info in result["cos_urls"]:
            original_syntax = url_info["original_info"]["markdown_syntax"]
            alt_text = url_info["original_info"]["alt_text"]
            new_syntax = f"![{alt_text}]({url_info['cos_url']})"
            processed_content = processed_content.replace(original_syntax, new_syntax)
        result["processed_content"] = processed_content

    return result

In [60]:
pip install pypdf python-docx python-pptx pillow requests

  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached python_pptx-1.0.2-py3-none-any.whl.metadata (2.5 kB)
  Using cached xlsxwriter-3.2.5-py3-none-any.whl.metadata (2.7 kB)
Using cached python_docx-1.2.0-py3-none-any.whl (252 kB)
Using cached python_pptx-1.0.2-py3-none-any.whl (472 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 11.3 MB/s  0:00:00 12.0 MB/s eta 0:00:01
Using cached xlsxwriter-3.2.5-py3-none-any.whl (172 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [python-pptx] 3/4 [python-pptx]
Note: you may need to restart the kernel to use updated packages.


In [69]:
import base64
import hashlib
import io
import logging
import os
import re
import zipfile
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from xml.etree import ElementTree as ET

from PIL import Image
from qcloud_cos import CosConfig, CosS3Client


class MultimodalDocumentLoader:
    """
    多模态文档加载器
    支持从PDF、Word、PowerPoint、Markdown等文档中提取图片，并上传到腾讯云COS
    """

    def __init__(self, cos_bucket: str = None, cos_region: str = None):
        """
        初始化多模态文档加载器

        Args:
            cos_bucket: COS存储桶名称（可选，从环境变量读取）
            cos_region: COS区域（可选，从环境变量读取）
        """
        # 加载COS配置
        try:
            load_dotenv()
            self.cos_secret_id = os.getenv("COS_SECRET_ID")
            self.cos_secret_key = os.getenv("COS_SECRET_KEY")
            self.cos_region = cos_region or os.getenv("COS_REGION")
            self.cos_bucket = cos_bucket or os.getenv("COS_BUCKET")
            self.cos_domain = os.getenv("COS_DOMAIN")

            print(f"🔧 COS配置加载成功: {self.cos_bucket} ({self.cos_region})")

            # 初始化COS客户端
            if CosConfig and CosS3Client:
                config = CosConfig(
                    Region=self.cos_region,
                    SecretId=self.cos_secret_id,
                    SecretKey=self.cos_secret_key,
                )
                self.cos_client = CosS3Client(config)
                self.cos_enabled = True
            else:
                self.cos_client = None
                self.cos_enabled = False
                print("⚠️ COS SDK未安装，将使用模拟上传模式")

        except Exception as e:
            print(f"⚠️ COS配置加载失败: {e}")
            print("将使用模拟上传模式，请检查.env文件中的COS配置")
            self.cos_enabled = False
            self.cos_client = None
            # 从环境变量中读取COS配置
            self.cos_bucket = cos_bucket or os.getenv("COS_BUCKET")
            self.cos_region = cos_region or os.getenv("COS_REGION")
            self.cos_domain = os.getenv("COS_DOMAIN")

        # 设置基础URL
        if hasattr(self, "cos_domain") and self.cos_domain:
            # 如果cos_domain已经包含协议，直接使用；否则添加https://
            if self.cos_domain.startswith(("http://", "https://")):
                self.base_url = self.cos_domain
            else:
                self.base_url = f"https://{self.cos_domain}"
        else:
            self.base_url = (
                f"https://{self.cos_bucket}.cos.{self.cos_region}.myqcloud.com"
            )

        # 创建临时图片存储目录
        self.temp_dir = Path("temp_images")
        self.temp_dir.mkdir(exist_ok=True)

        print(f"多模态文档加载器初始化完成，COS配置: {self.base_url}")

    def extract_images_from_pdf(self, pdf_path: str) -> List[Dict]:
        """
        从PDF文件中提取图片

        Args:
            pdf_path: PDF文件路径

        Returns:
            图片信息列表，包含图片数据、页码、位置等
        """
        images = []

        try:
            # 使用PyMuPDF打开PDF
            doc = fitz.open(pdf_path)
            logger.info(f"开始从PDF提取图片: {pdf_path}，共{len(doc)}页")

            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                image_list = page.get_images(full=True)

                for img_index, img in enumerate(image_list):
                    # 获取图片对象
                    xref = img[0]
                    pix = fitz.Pixmap(doc, xref)

                    # 转换为PIL Image
                    if pix.n - pix.alpha < 4:  # GRAY or RGB
                        img_data = pix.tobytes("png")
                        img_pil = Image.open(io.BytesIO(img_data))

                        # 生成图片哈希作为文件名
                        img_hash = hashlib.md5(img_data).hexdigest()

                        images.append(
                            {
                                "data": img_data,
                                "format": "png",
                                "page": page_num + 1,
                                "index": img_index,
                                "hash": img_hash,
                                "size": img_pil.size,
                                "filename": f"pdf_page{page_num+1}_img{img_index}_{img_hash[:8]}.png",
                            }
                        )

                        print(
                            f"提取图片: 页面{page_num+1}, 索引{img_index}, 尺寸{img_pil.size}"
                        )

                    pix = None  # 释放内存

            doc.close()
            print(f"PDF图片提取完成，共提取{len(images)}张图片")

        except Exception as e:
            print(f"PDF图片提取失败: {e}")

        return images

    def extract_images_from_word(self, word_path: str) -> List[Dict]:
        """
        从Word文档中提取图片

        Args:
            word_path: Word文档路径

        Returns:
            图片信息列表
        """
        images = []

        try:
            logger.info(f"开始从Word文档提取图片: {word_path}")

            # Word文档实际上是一个ZIP文件
            with zipfile.ZipFile(word_path, "r") as docx:
                # 查找媒体文件
                media_files = [
                    f for f in docx.namelist() if f.startswith("word/media/")
                ]

                for media_file in media_files:
                    # 读取图片数据
                    img_data = docx.read(media_file)

                    # 获取文件扩展名
                    file_ext = Path(media_file).suffix.lower()
                    if file_ext in [".png", ".jpg", ".jpeg", ".gif", ".bmp"]:
                        # 生成图片哈希
                        img_hash = hashlib.md5(img_data).hexdigest()

                        # 获取图片尺寸
                        try:
                            img_pil = Image.open(io.BytesIO(img_data))
                            size = img_pil.size
                        except:
                            size = (0, 0)

                        images.append(
                            {
                                "data": img_data,
                                "format": file_ext[1:],  # 去掉点号
                                "original_path": media_file,
                                "hash": img_hash,
                                "size": size,
                                "filename": f"word_{Path(media_file).stem}_{img_hash[:8]}{file_ext}",
                            }
                        )

                        print(f"提取图片: {media_file}, 尺寸{size}")

            print(f"Word图片提取完成，共提取{len(images)}张图片")

        except Exception as e:
            print(f"Word图片提取失败: {e}")

        return images

    def extract_images_from_ppt(self, ppt_path: str) -> List[Dict]:
        """
        从PowerPoint文件中提取图片

        Args:
            ppt_path: PowerPoint文件路径

        Returns:
            图片信息列表
        """
        images = []

        try:
            if not Presentation:
                print("python-pptx库未安装，无法处理PPT文件")
                return images

            print(f"开始从PowerPoint文件提取图片: {ppt_path}")

            # 打开PowerPoint文件
            prs = Presentation(ppt_path)

            for slide_num, slide in enumerate(prs.slides):
                for shape_num, shape in enumerate(slide.shapes):
                    # 检查是否为图片形状
                    if hasattr(shape, "image") and shape.image:
                        try:
                            # 获取图片数据
                            img_data = shape.image.blob

                            # 获取图片格式
                            img_format = shape.image.ext
                            if not img_format.startswith("."):
                                img_format = "." + img_format

                            # 生成图片哈希
                            img_hash = hashlib.md5(img_data).hexdigest()

                            # 获取图片尺寸
                            try:
                                img_pil = Image.open(io.BytesIO(img_data))
                                size = img_pil.size
                            except:
                                size = (0, 0)

                            images.append(
                                {
                                    "data": img_data,
                                    "format": (
                                        img_format[1:]
                                        if img_format.startswith(".")
                                        else img_format
                                    ),
                                    "slide": slide_num + 1,
                                    "shape": shape_num,
                                    "hash": img_hash,
                                    "size": size,
                                    "filename": f"ppt_slide{slide_num+1}_shape{shape_num}_{img_hash[:8]}{img_format}",
                                }
                            )

                            print(
                                f"提取图片: 幻灯片{slide_num+1}, 形状{shape_num}, 尺寸{size}"
                            )

                        except Exception as e:
                            print(
                                f"提取幻灯片{slide_num+1}形状{shape_num}图片失败: {e}"
                            )
                            continue

            print(f"PowerPoint图片提取完成，共提取{len(images)}张图片")

        except Exception as e:
            print(f"PowerPoint图片提取失败: {e}")

        return images

    def extract_images_from_markdown(self, md_path: str) -> List[Dict]:
        """
        从Markdown文件中提取图片引用

        Args:
            md_path: Markdown文件路径

        Returns:
            图片信息列表
        """
        images = []

        try:
            print(f"开始从Markdown文件提取图片引用: {md_path}")

            with open(md_path, "r", encoding="utf-8") as f:
                content = f.read()

            # 匹配Markdown图片语法: ![alt](path)
            img_pattern = r"!\[([^\]]*)\]\(([^\)]+)\)"
            matches = re.finditer(img_pattern, content)

            md_dir = Path(md_path).parent

            for match in matches:
                alt_text = match.group(1)
                img_path = match.group(2)

                # 如果是相对路径，转换为绝对路径
                if not img_path.startswith(("http://", "https://")):
                    full_img_path = md_dir / img_path

                    if full_img_path.exists():
                        # 读取本地图片文件
                        with open(full_img_path, "rb") as img_file:
                            img_data = img_file.read()

                        # 获取图片格式
                        file_ext = full_img_path.suffix.lower()

                        # 生成图片哈希
                        img_hash = hashlib.md5(img_data).hexdigest()

                        # 获取图片尺寸
                        try:
                            img_pil = Image.open(io.BytesIO(img_data))
                            size = img_pil.size
                        except:
                            size = (0, 0)

                        images.append(
                            {
                                "data": img_data,
                                "format": file_ext[1:] if file_ext else "png",
                                "alt_text": alt_text,
                                "original_path": img_path,
                                "full_path": str(full_img_path),
                                "hash": img_hash,
                                "size": size,
                                "filename": f"md_{full_img_path.stem}_{img_hash[:8]}{file_ext}",
                                "markdown_syntax": match.group(0),
                            }
                        )

                        print(f"提取图片: {img_path}, 尺寸{size}")
                    else:
                        print(f"图片文件不存在: {full_img_path}")

            print(f"Markdown图片提取完成，共提取{len(images)}张图片")

        except Exception as e:
            print(f"Markdown图片提取失败: {e}")

        return images

    def upload_to_cos(self, image_data: bytes, filename: str) -> str:
        """
        上传图片到腾讯云COS

        Args:
            image_data: 图片二进制数据
            filename: 文件名

        Returns:
            COS URL或模拟URL
        """
        try:
            if self.cos_enabled and self.cos_client:
                # 真实上传到COS
                object_key = f"images/{filename}"

                # 上传文件
                response = self.cos_client.put_object(
                    Bucket=self.cos_bucket,
                    Body=image_data,
                    Key=object_key,
                    ContentType=self._get_content_type(filename),
                )

                # 生成访问URL
                cos_url = f"{self.base_url}/{object_key}"

                print(f"✅ COS上传成功: {filename} -> {cos_url}")
                return cos_url

            else:
                # 模拟上传模式
                return self._simulate_cos_upload(image_data, filename)

        except Exception as e:
            print(f"❌ COS上传失败: {e}")
            # 失败时回退到模拟模式
            return self._simulate_cos_upload(image_data, filename)

    def _simulate_cos_upload(self, image_data: bytes, filename: str) -> str:
        """
        模拟上传图片到腾讯云COS

        Args:
            image_data: 图片二进制数据
            filename: 文件名

        Returns:
            模拟的COS URL
        """
        try:
            # 保存到临时目录（模拟上传）
            temp_file = self.temp_dir / filename
            with open(temp_file, "wb") as f:
                f.write(image_data)

            # 生成模拟的COS URL
            cos_url = f"{self.base_url}/images/{filename}"

            print(f"🔄 模拟上传: {filename} -> {cos_url}")
            return cos_url

        except Exception as e:
            print(f"❌ 模拟上传失败: {e}")
            return ""

    def _get_content_type(self, filename: str) -> str:
        """
        根据文件扩展名获取Content-Type

        Args:
            filename: 文件名

        Returns:
            Content-Type字符串
        """
        ext = Path(filename).suffix.lower()
        content_types = {
            ".jpg": "image/jpeg",
            ".jpeg": "image/jpeg",
            ".png": "image/png",
            ".gif": "image/gif",
            ".bmp": "image/bmp",
            ".webp": "image/webp",
            ".svg": "image/svg+xml",
            ".pptx": "application/vnd.openxmlformats-officedocument.presentationml.presentation",
            ".ppt": "application/vnd.ms-powerpoint",
        }
        return content_types.get(ext, "application/octet-stream")

    def process_document(self, file_path: str) -> Dict:
        """
        处理文档，提取图片并替换为COS URL

        Args:
            file_path: 文档文件路径

        Returns:
            处理结果，包含原始内容、处理后内容、图片信息等
        """
        file_path = Path(file_path)
        file_ext = file_path.suffix.lower()

        print(f"开始处理文档: {file_path}")

        result = {
            "file_path": str(file_path),
            "file_type": file_ext,
            "images": [],
            "cos_urls": [],
            "original_content": "",
            "processed_content": "",
            "success": False,
        }

        try:
            # 根据文件类型提取图片
            if file_ext == ".pdf":
                images = self.extract_images_from_pdf(str(file_path))
            elif file_ext in [".docx", ".doc"]:
                images = self.extract_images_from_word(str(file_path))
            elif file_ext in [".pptx", ".ppt"]:
                images = self.extract_images_from_ppt(str(file_path))
            elif file_ext in [".md", ".markdown"]:
                images = self.extract_images_from_markdown(str(file_path))
                # 读取原始Markdown内容
                with open(file_path, "r", encoding="utf-8") as f:
                    result["original_content"] = f.read()
            else:
                print(f"不支持的文件类型: {file_ext}")
                return result

            result["images"] = images

            # 上传图片到COS并获取URL
            cos_urls = []
            for img in images:
                cos_url = self.upload_to_cos(img["data"], img["filename"])
                if cos_url:
                    cos_urls.append(
                        {
                            "filename": img["filename"],
                            "cos_url": cos_url,
                            "original_info": img,
                        }
                    )

            result["cos_urls"] = cos_urls

            # 如果是Markdown文件，替换图片引用
            if file_ext in [".md", ".markdown"] and result["original_content"]:
                processed_content = result["original_content"]

                for i, img in enumerate(images):
                    if i < len(cos_urls):
                        # 替换Markdown图片语法
                        old_syntax = img["markdown_syntax"]
                        new_syntax = f"![{img['alt_text']}]({cos_urls[i]['cos_url']})"
                        processed_content = processed_content.replace(
                            old_syntax, new_syntax
                        )

                result["processed_content"] = processed_content

            result["success"] = True
            print(f"文档处理完成: 提取{len(images)}张图片，上传{len(cos_urls)}个URL")

        except Exception as e:
            logger.error(f"文档处理失败: {e}")
            result["error"] = str(e)

        return result

    def cleanup_temp_files(self):
        """清理临时文件"""
        try:
            import shutil

            if self.temp_dir.exists():
                shutil.rmtree(self.temp_dir)
                print("临时文件清理完成")
        except Exception as e:
            print(f"临时文件清理失败: {e}")

ModuleNotFoundError: No module named 'qcloud_cos'

In [68]:
loader = MultimodalDocumentLoader()
pdf_path = "docs/ai_model_agent_history_script.pdf"
result = loader.process_document(pdf_path)

if result.get("success", False):
    print("✅ PDF 处理成功！")
    print(f"🖼️  共提取图片: {len(result['images'])} 张")
    print(f"☁️  成功上传并获取 URL: {len(result['cos_urls'])} 个\n")
    print("前2个图片的云端URL示例:")
    for i, url_info in enumerate(result["cos_urls"][:2]):
        print(f"   - 图片 {i+1}: {url_info['cos_url']}")
else:
    print(f"❌ 处理失败: {result.get('error')}")

🔧 COS配置加载成功: langchain-1300167186 (ap-guangzhou)
⚠️ COS SDK未安装，将使用模拟上传模式
多模态文档加载器初始化完成，COS配置: https://langchain-1300167186.cos.ap-guangzhou.myqcloud.com
开始处理文档: docs/ai_model_agent_history_script.pdf
PDF图片提取失败: name 'logger' is not defined
文档处理完成: 提取0张图片，上传0个URL
✅ PDF 处理成功！
🖼️  共提取图片: 0 张
☁️  成功上传并获取 URL: 0 个

前2个图片的云端URL示例:


In [58]:
md_path = "docs/test_markdown.md"
result = loader.process_document(md_path)

# 打印处理前后的内容对比
if result.get("success", False):
    print(f"✅ Markdown 处理成功！")
    print("\n" + "=" * 20 + " 原始 Markdown 内容 " + "=" * 20 + "\n")
    print(result["original_content"])
    print("\n" + "=" * 20 + " 处理后的 Markdown 内容 " + "=" * 20 + "\n")
    print(result["processed_content"])
else:
    print(f"❌ 处理失败: {result.get('error')}")

❌ 处理失败: 暂不支持的文件类型: .md
